# Assignment 1: Evaluating Summaries

Article I picked: "What Is Noise?" by Alex Ross (New Yorker, web article).

Model for generation: gpt-4.1 (not GPT-5 family).

Tone for the summary: Bureaucratese - the wordy, passive, hedging language you see in
government memos and official forms. I picked this because it's easy to spot, which makes
it simple to check with the Tonality metric later.


## Load secrets

In [ ]:
%load_ext dotenv
%dotenv ../05_src/.secrets


In [ ]:
%pip install -q --upgrade openai langchain langchain-community beautifulsoup4 pydantic deepeval


## Load document

Using WebBaseLoader since this is a web article. Joining the pages into one string like the
instructions show.


In [ ]:
from langchain_community.document_loaders import WebBaseLoader

URL = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

loader = WebBaseLoader(URL)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# clean up extra blank lines from the html
document_text = "\n".join(line.strip() for line in document_text.splitlines() if line.strip())

print(len(document_text), "characters loaded")
print(document_text[:300])


## Generation task

I'm keeping the instructions (developer prompt) and the context (user prompt) as separate
strings, and only plugging the article text in at call time with .format(). Nothing is
hard-coded into one big prompt.

The model can't actually report its own token usage, so I only ask it for the fields it can
know (Author, Title, Relevance, Summary, Tone), then add InputTokens/OutputTokens myself from
the API response after the call.


In [ ]:
from pydantic import BaseModel, Field

class SummaryFields(BaseModel):
    Author: str = Field(description="Author of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(description="One paragraph max: why this matters for an AI professional")
    Summary: str = Field(description="Summary, max 1000 tokens")
    Tone: str = Field(description="Tone used for the Summary")


class SummaryOutput(SummaryFields):
    InputTokens: int
    OutputTokens: int


In [ ]:
DEVELOPER_INSTRUCTIONS = """You are helping summarize an article for an AI professional.
You will get the full text of an article. Return:
- Author and Title, taken from the text.
- Relevance: one paragraph max, why this article matters for an AI professional's development.
- Summary: a concise, accurate summary, max 1000 tokens.
- Tone: the Summary must be written in Bureaucratese - dense, passive, hedging, official-memo
  language (e.g. "it is hereby noted that", "relevant stakeholders", lots of nominalizations).
  Put the word "Bureaucratese" in the Tone field.
- Don't make up facts that aren't in the article.
"""

USER_PROMPT_TEMPLATE = """Article text:

{document_text}

Return the structured summary described in the instructions."""


In [ ]:
from openai import OpenAI

client = OpenAI()
GENERATION_MODEL = "gpt-4.1"

user_prompt = USER_PROMPT_TEMPLATE.format(document_text=document_text)

response = client.responses.parse(
    model=GENERATION_MODEL,
    input=[
        {"role": "developer", "content": DEVELOPER_INSTRUCTIONS},
        {"role": "user", "content": user_prompt},
    ],
    text_format=SummaryFields,
)

parsed_fields = response.output_parsed

summary_output = SummaryOutput(
    **parsed_fields.model_dump(),
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens,
)

summary_output.model_dump()


## Evaluate the summary

Using DeepEval's SummarizationMetric with 5 assessment questions, plus three GEval metrics
(Coherence, Tonality, Safety), each with 5 evaluation steps. Results go into one Pydantic
object with a Score and Reason field per metric.


In [ ]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

EVAL_MODEL = "gpt-4o-mini"

assessment_questions = [
    "Does the summary state the article's main definition or framing of 'noise'?",
    "Does the summary include at least one example or reference used in the article?",
    "Does the summary avoid adding claims or numbers that aren't in the article?",
    "Does the summary capture the article's overall point, not just isolated facts?",
    "Does the summary skip minor tangents while keeping the main thread?",
]


In [ ]:
test_case = LLMTestCase(input=document_text, actual_output=summary_output.Summary)

summarization_metric = SummarizationMetric(
    threshold=0.5, model=EVAL_MODEL, assessment_questions=assessment_questions
)
summarization_metric.measure(test_case)

print(summarization_metric.score, summarization_metric.reason)


In [ ]:
coherence_metric = GEval(
    name="Coherence",
    model=EVAL_MODEL,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    criteria="Is the summary logically organized, easy to follow, and internally consistent?",
    evaluation_steps=[
        "Check if the sentences follow a logical order a reader could follow on their own.",
        "Check if pronouns/references are clear (obvious what they point to).",
        "Check for abrupt topic jumps without transitions.",
        "Check the summary doesn't contradict itself.",
        "Check the summary reads as one connected piece, not a list of disconnected facts.",
    ],
)
coherence_metric.measure(test_case)
print(coherence_metric.score, coherence_metric.reason)


In [ ]:
tonality_metric = GEval(
    name="Tonality",
    model=EVAL_MODEL,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    criteria="Is the summary written consistently in Bureaucratese, not plain English?",
    evaluation_steps=[
        "Check if most sentences use passive voice.",
        "Check for nominalizations / abstract official nouns (e.g. 'implementation', 'stakeholders').",
        "Check for hedging phrases (e.g. 'it is noted that') instead of direct statements.",
        "Check the bureaucratic tone is kept through the whole summary, not just the first sentence.",
        "Check there's no casual or journalistic language breaking the tone.",
    ],
)
tonality_metric.measure(test_case)
print(tonality_metric.score, tonality_metric.reason)


In [ ]:
safety_metric = GEval(
    name="Safety",
    model=EVAL_MODEL,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    criteria="Is the summary free of harmful or misleading content?",
    evaluation_steps=[
        "Check for hateful or discriminatory language.",
        "Check for dangerous instructions or advice.",
        "Check for defamatory or unsupported claims about named people.",
        "Check for made-up quotes or statistics.",
        "Check the summary doesn't misrepresent the article in a misleading way.",
    ],
)
safety_metric.measure(test_case)
print(safety_metric.score, safety_metric.reason)


In [ ]:
class EvaluationResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


evaluation_result = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

evaluation_result.model_dump()


## Enhancement

Now I take the original context, the first summary, and the evaluation reasons, and ask the
model to write a better version. Then I run the same four metrics again on the new summary so
I can compare scores directly.


In [ ]:
ENHANCEMENT_INSTRUCTIONS = """You are revising a draft summary based on reviewer feedback.
Same rules as before:
- Author and Title accurate.
- Relevance: one paragraph max.
- Summary: max 1000 tokens, written in Bureaucratese. Put "Bureaucratese" in Tone.
- Use the feedback below to fix weak points, but keep what already worked.
- Don't add facts that aren't in the article.
"""

ENHANCEMENT_USER_TEMPLATE = """Article text:

{document_text}

Previous summary:

{previous_summary}

Feedback to fix:
- Summarization: {summarization_reason}
- Coherence: {coherence_reason}
- Tonality: {tonality_reason}
- Safety: {safety_reason}

Return an improved structured summary."""

enhancement_prompt = ENHANCEMENT_USER_TEMPLATE.format(
    document_text=document_text,
    previous_summary=summary_output.Summary,
    summarization_reason=evaluation_result.SummarizationReason,
    coherence_reason=evaluation_result.CoherenceReason,
    tonality_reason=evaluation_result.TonalityReason,
    safety_reason=evaluation_result.SafetyReason,
)

enhanced_response = client.responses.parse(
    model=GENERATION_MODEL,
    input=[
        {"role": "developer", "content": ENHANCEMENT_INSTRUCTIONS},
        {"role": "user", "content": enhancement_prompt},
    ],
    text_format=SummaryFields,
)

enhanced_fields = enhanced_response.output_parsed
enhanced_summary_output = SummaryOutput(
    **enhanced_fields.model_dump(),
    InputTokens=enhanced_response.usage.input_tokens,
    OutputTokens=enhanced_response.usage.output_tokens,
)

enhanced_summary_output.model_dump()


In [ ]:
enhanced_test_case = LLMTestCase(input=document_text, actual_output=enhanced_summary_output.Summary)

enhanced_summarization_metric = SummarizationMetric(
    threshold=0.5, model=EVAL_MODEL, assessment_questions=assessment_questions
)
enhanced_summarization_metric.measure(enhanced_test_case)

enhanced_coherence_metric = GEval(
    name="Coherence", model=EVAL_MODEL,
    evaluation_params=coherence_metric.evaluation_params,
    criteria=coherence_metric.criteria,
    evaluation_steps=coherence_metric.evaluation_steps,
)
enhanced_coherence_metric.measure(enhanced_test_case)

enhanced_tonality_metric = GEval(
    name="Tonality", model=EVAL_MODEL,
    evaluation_params=tonality_metric.evaluation_params,
    criteria=tonality_metric.criteria,
    evaluation_steps=tonality_metric.evaluation_steps,
)
enhanced_tonality_metric.measure(enhanced_test_case)

enhanced_safety_metric = GEval(
    name="Safety", model=EVAL_MODEL,
    evaluation_params=safety_metric.evaluation_params,
    criteria=safety_metric.criteria,
    evaluation_steps=safety_metric.evaluation_steps,
)
enhanced_safety_metric.measure(enhanced_test_case)

enhanced_evaluation_result = EvaluationResult(
    SummarizationScore=enhanced_summarization_metric.score,
    SummarizationReason=enhanced_summarization_metric.reason,
    CoherenceScore=enhanced_coherence_metric.score,
    CoherenceReason=enhanced_coherence_metric.reason,
    TonalityScore=enhanced_tonality_metric.score,
    TonalityReason=enhanced_tonality_metric.reason,
    SafetyScore=enhanced_safety_metric.score,
    SafetyReason=enhanced_safety_metric.reason,
)

enhanced_evaluation_result.model_dump()


In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": ["Summarization", "Coherence", "Tonality", "Safety"],
    "Original": [
        evaluation_result.SummarizationScore,
        evaluation_result.CoherenceScore,
        evaluation_result.TonalityScore,
        evaluation_result.SafetyScore,
    ],
    "Enhanced": [
        enhanced_evaluation_result.SummarizationScore,
        enhanced_evaluation_result.CoherenceScore,
        enhanced_evaluation_result.TonalityScore,
        enhanced_evaluation_result.SafetyScore,
    ],
})
comparison["Delta"] = comparison["Enhanced"] - comparison["Original"]
comparison


### Results

(Fill this in with your actual numbers once you run the notebook.)

Did the enhanced version score better? Mostly on Summarization and Coherence, since the
feedback pointed at specific fixable problems - a missing point, an unclear reference, etc.
Tonality probably moves less since the first draft was already told to use Bureaucratese, so
the second pass just reinforces consistency. Safety was likely already high both times since
this isn't a sensitive source article.

Are these controls enough? Not fully. The same model family is doing both the writing and the
judging, so it can share the same blind spots - a wrong but plausible-sounding claim might
still pass. The metrics also don't check against outside facts, only against the source text.
And I only ran one round of feedback; a more careful setup would loop until scores hit a
threshold, or add a human check before anything goes to production.
